In [ ]:
import scanpy as sc
import numpy as np
import pandas as pd
import omicverse as ov
from nsf import sim

In [ ]:

import numpy as np
from anndata import AnnData

np.random.seed(101)

def generate_simulated_dataset(scenario="quilt", nside=36, nzprob_nsp=0.2, bkg_mean=0.2, nb_shape=10.0, seed=101):
    """
    Generate a simulated spatial transcriptomics dataset.

    Parameters:
    scenario (str): Simulation scenario, can be "quilt", "ggblocks", or "both", default is "quilt".
    nside (int): Side length of the spatial grid, default is 36.
    nzprob_nsp (float): Probability of non-zero elements in non-spatial factors, default is 0.2.
    bkg_mean (float): Mean of the background negative binomial distribution, default is 0.2.
    nb_shape (float): Shape parameter of the negative binomial distribution, default is 10.0.
    seed (int): Random seed, default is 101.

    Returns:
    AnnData object: An AnnData object containing the simulated dataset.
    """
    adata = sim.sim(scenario=scenario, nside=nside, nzprob_nsp=nzprob_nsp, bkg_mean=bkg_mean, nb_shape=nb_shape, seed=seed)
    
    return adata


# Simulated_data_1

In [ ]:

adata_rna = sc.read_h5ad('Original_SingleCell_Multiomics_Data/Dataset_1/Chen-2019-RNA.h5ad')
adata_rna.layers['raw'] = adata_rna.X
adata_rna=ov.pp.preprocess(adata_rna,mode='shiftlog|pearson',n_HVGs=3000,
                       target_sum=50*1e4)
adata_rna.raw = adata_rna
adata_rna = adata_rna[:, adata_rna.var.highly_variable_features]
ov.pp.scale(adata_rna)
ov.pp.pca(adata_rna,layer='scaled',n_pcs=50)
ov.pp.neighbors(adata_rna, n_neighbors=15, n_pcs=50,
               use_rep='scaled|original|X_pca')
ov.pp.umap(adata_rna)


In [ ]:
from matplotlib import patheffects
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(4,4))

ov.pl.embedding(adata_rna,
                  basis='X_umap',
                  color=['cell_type'],
                   show=False, legend_loc=None, add_outline=False, 
                   frameon='small',legend_fontoutline=2,ax=ax
                 )

ov.utils.gen_mpl_labels(
    adata_rna,
    'cell_type',
    exclude=("None",),  
    basis='X_umap',
    ax=ax,
    adjust_kwargs=dict(arrowprops=dict(arrowstyle='-', color='black')),
    text_kwargs=dict(fontsize= 12 ,weight='bold',
                     path_effects=[patheffects.withStroke(linewidth=2, foreground='w')] ),
)

In [ ]:
for celltype in set(adata_rna.obs['cell_type']):
    print(f"The number of cell type {celltype} is {len(adata_rna[adata_rna.obs['cell_type']==celltype].obs_names)}.")

In [ ]:
simulated_dataset = generate_simulated_dataset(scenario="quilt", nside=36, nzprob_nsp=0.2, bkg_mean=0.2, nb_shape=10.0, seed=101)

print(simulated_dataset)

In [ ]:
import numpy as np
import pandas as pd

max_indices = np.argmax(simulated_dataset.obsm['spfac'], axis=1)

series = pd.Series(max_indices)

simulated_dataset.obs['spatial_domain'] = max_indices.tolist()
simulated_dataset.obs['spatial_domain'] = simulated_dataset.obs['spatial_domain'].astype('category')

sc.pl.spatial(simulated_dataset,color=['spatial_domain'],spot_size=0.1)

In [ ]:
for spot in set(simulated_dataset.obs['spatial_domain']):
    print(f"The number of spatial domain {spot} is {len(simulated_dataset[simulated_dataset.obs['spatial_domain']==spot].obs_names)}.")

In [ ]:
import scanpy as sc
import numpy as np

np.random.seed(101)


# Specify the cell types to sample and the number of cells for each type
num_cells_per_celltype = {
    'E2Rasgrf2': 648,
    'E3Rorb': 396,
    'E5Galnt14': 36,
    'E4Il1rapl2': 216
}

# Mapping of cluster labels to cell type annotations
cluster2annotation = {
    0: 'E2Rasgrf2',
    1: 'E3Rorb',
    2: 'E5Galnt14',
    3: 'E4Il1rapl2'
}

# Assign cell types to the simulated dataset based on cluster labels
simulated_dataset.obs['cell_type'] = simulated_dataset.obs['spatial_domain'].map(cluster2annotation).astype('category')

# Initialize an empty list to store the indices of selected cells
selected_indices = []

# Create a dictionary to store spatial coordinates for each cell type
celltype_coords = {}

# Iterate over each cell type and randomly sample the specified number of cells
for cell_type in num_cells_per_celltype.keys():
    # Get the indices of cells belonging to the current cell type
    indices = np.where(adata_rna.obs['cell_type'] == cell_type)[0]
    
    # Randomly sample the specified number of cell indices
    selected_indices.extend(np.random.choice(indices, num_cells_per_celltype[cell_type], replace=False))

    # Get the spatial coordinates for cells of the current cell type
    coords = simulated_dataset[simulated_dataset.obs['cell_type'] == cell_type].obsm['spatial']
    celltype_coords[cell_type] = coords

# Initialize an empty array to store the mapped spatial coordinates
mapped_coords = np.zeros((simulated_dataset.n_obs, 2))

# Convert the selected cell indices to a boolean mask
selected_mask = np.in1d(np.arange(adata_rna.n_obs), selected_indices)

# Create a new AnnData object based on the selected cells
new_adata = adata_rna[selected_mask]

# Iterate over the cells in the new AnnData object and map their spatial coordinates
celltype_index_counters = {cell_type: 0 for cell_type in simulated_dataset.obs['cell_type'].unique()}

for i, cell_type in enumerate(new_adata.obs['cell_type']):
    index = celltype_index_counters[cell_type]
    mapped_coords[i] = celltype_coords[cell_type][index]
    celltype_index_counters[cell_type] += 1

# Add the mapped spatial coordinates to the new AnnData object's obsm
new_adata.obsm['spatial'] = mapped_coords

# Create a new AnnData object with the mapped spatial coordinates and UMAP embeddings
adata_rna_new = new_adata
adata_rna_new.obsm['X_umap'] = adata_rna[adata_rna_new.obs_names, :].obsm['X_umap']

In [ ]:
sc.pl.spatial(adata_rna_new,color=['cell_type'],spot_size=0.12)

In [ ]:
from matplotlib import patheffects
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(4,4))

ov.pl.embedding(adata_rna_new,
                  basis='X_umap',
                  color=['cell_type'],
                   show=False, legend_loc=None, add_outline=False, 
                   frameon='small',legend_fontoutline=2,ax=ax
                 )

ov.utils.gen_mpl_labels(
    adata_rna_new,
    'cell_type',
    exclude=("None",),  
    basis='X_umap',
    ax=ax,
    adjust_kwargs=dict(arrowprops=dict(arrowstyle='-', color='black')),
    text_kwargs=dict(fontsize= 12 ,weight='bold',
                     path_effects=[patheffects.withStroke(linewidth=2, foreground='w')] ),
)

In [ ]:
adata_rna_new

In [ ]:
adata_rna_new.obs_names = [s[:-3] for s in adata_rna_new.obs_names]

adata_atac = sc.read_h5ad('Original_SingleCell_Multiomics_Data/Dataset_1/Chen-2019-ATAC.h5ad')
adata_atac.obs_names = [s[:-4] for s in adata_atac.obs_names]
adata_atac_new = adata_atac[adata_rna_new.obs_names,:]
adata_atac_new.obs

In [ ]:
adata_rna_new.write_h5ad('Original_Simulated_Data/Simulated_Dataset_1/SimulatedData_1_rna.h5ad',compression='gzip')
adata_atac_new.write_h5ad('Original_Simulated_Data/Simulated_Dataset_1/SimulatedData_1_atac.h5ad',compression='gzip')

# Simulated_data_2

In [ ]:
adata_rna = sc.read_h5ad('Original_SingleCell_Multiomics_Data/Dataset_2/cd34_multiome_rna.h5ad')
adata_rna.X.max()

In [ ]:
adata_rna.layers['raw'] = adata_rna.X
adata_rna=ov.pp.preprocess(adata_rna,mode='shiftlog|pearson',n_HVGs=3000,
                       target_sum=50*1e4)
adata_rna.raw = adata_rna
adata_rna = adata_rna[:, adata_rna.var.highly_variable_features]
ov.pp.scale(adata_rna)
ov.pp.pca(adata_rna,layer='scaled',n_pcs=50)
adata_rna

In [ ]:
from matplotlib import patheffects
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(4,4))

ov.pl.embedding(adata_rna,
                  basis='X_umap',
                  color=['celltype'],
                   show=False, legend_loc=None, add_outline=False, 
                   frameon='small',legend_fontoutline=2,ax=ax
                 )

ov.utils.gen_mpl_labels(
    adata_rna,
    'celltype',
    exclude=("None",),  
    basis='X_umap',
    ax=ax,
    adjust_kwargs=dict(arrowprops=dict(arrowstyle='-', color='black')),
    text_kwargs=dict(fontsize= 12 ,weight='bold',
                     path_effects=[patheffects.withStroke(linewidth=2, foreground='w')] ),
)

In [ ]:
for celltype in set(adata_rna.obs['celltype']):
    print(f"The number of cell type {celltype} is {len(adata_rna[adata_rna.obs['celltype']==celltype].obs_names)}.")

In [ ]:
simulated_dataset = generate_simulated_dataset(scenario="quilt", nside=36, nzprob_nsp=0.2, bkg_mean=0.2, nb_shape=10.0, seed=101)

import numpy as np
import pandas as pd

max_indices = np.argmax(simulated_dataset.obsm['spfac'], axis=1)

series = pd.Series(max_indices)

simulated_dataset.obs['spatial_domain'] = max_indices.tolist()
simulated_dataset.obs['spatial_domain'] = simulated_dataset.obs['spatial_domain'].astype('category')

sc.pl.spatial(simulated_dataset,color=['spatial_domain'],spot_size=0.1)

In [ ]:
import scanpy as sc
import numpy as np

np.random.seed(101)


# Specify the cell types to sample and the number of cells for each type
num_cells_per_celltype = {
    'HSC': 648,
    'HMP': 396,
    'CLP': 36,
    'Mono': 216
}

# Mapping of cluster labels to cell type annotations
cluster2annotation = {
    0: 'HSC',
    1: 'HMP',
    2: 'CLP',
    3: 'Mono'
}

# Assign cell types to the simulated dataset based on cluster labels
simulated_dataset.obs['cell_type'] = simulated_dataset.obs['spatial_domain'].map(cluster2annotation).astype('category')

# Initialize an empty list to store the indices of selected cells
selected_indices = []

# Create a dictionary to store spatial coordinates for each cell type
celltype_coords = {}

# Iterate over each cell type and randomly sample the specified number of cells
for cell_type in num_cells_per_celltype.keys():
    # Get the indices of cells belonging to the current cell type
    indices = np.where(adata_rna.obs['celltype'] == cell_type)[0]
    
    # Randomly sample the specified number of cell indices
    selected_indices.extend(np.random.choice(indices, num_cells_per_celltype[cell_type], replace=False))

    # Get the spatial coordinates for cells of the current cell type
    coords = simulated_dataset[simulated_dataset.obs['cell_type'] == cell_type].obsm['spatial']
    celltype_coords[cell_type] = coords

# Initialize an empty array to store the mapped spatial coordinates
mapped_coords = np.zeros((simulated_dataset.n_obs, 2))

# Convert the selected cell indices to a boolean mask
selected_mask = np.in1d(np.arange(adata_rna.n_obs), selected_indices)

# Create a new AnnData object based on the selected cells
new_adata = adata_rna[selected_mask]

# Iterate over the cells in the new AnnData object and map their spatial coordinates
celltype_index_counters = {cell_type: 0 for cell_type in simulated_dataset.obs['cell_type'].unique()}

for i, cell_type in enumerate(new_adata.obs['celltype']):
    index = celltype_index_counters[cell_type]
    mapped_coords[i] = celltype_coords[cell_type][index]
    celltype_index_counters[cell_type] += 1

# Add the mapped spatial coordinates to the new AnnData object's obsm
new_adata.obsm['spatial'] = mapped_coords

# Create a new AnnData object with the mapped spatial coordinates and UMAP embeddings
adata_rna_new = new_adata
adata_rna_new.obsm['X_umap'] = adata_rna[adata_rna_new.obs_names, :].obsm['X_umap']
adata_rna_new.obs['cell_type'] = adata_rna_new.obs['celltype']
adata_rna_new

In [ ]:
from matplotlib import patheffects
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(4,4))

ov.pl.embedding(adata_rna_new,
                  basis='X_umap',
                  color=['cell_type'],
                   show=False, legend_loc=None, add_outline=False, 
                   frameon='small',legend_fontoutline=2,ax=ax
                 )

ov.utils.gen_mpl_labels(
    adata_rna_new,
    'cell_type',
    exclude=("None",),  
    basis='X_umap',
    ax=ax,
    adjust_kwargs=dict(arrowprops=dict(arrowstyle='-', color='black')),
    text_kwargs=dict(fontsize= 12 ,weight='bold',
                     path_effects=[patheffects.withStroke(linewidth=2, foreground='w')] ),
)

In [ ]:
adata_atac = sc.read_h5ad('Original_SingleCell_Multiomics_Data/Dataset_2/cd34_multiome_atac.h5ad')
adata_atac_new = adata_atac[adata_rna_new.obs_names,:]
adata_atac_new.obs

In [ ]:
adata_rna_new.write_h5ad('Original_Simulated_Data/Simulated_Dataset_2/SimulatedData_2_rna.h5ad',compression='gzip')
adata_atac_new.write_h5ad('Original_Simulated_Data/Simulated_Dataset_2/SimulatedData_2_atac.h5ad',compression='gzip')

# Simulated_data_3

In [ ]:

adata_rna = sc.read_h5ad('Original_SingleCell_Multiomics_Data/Dataset_1/Chen-2019-RNA.h5ad')
adata_rna.layers['raw'] = adata_rna.X
adata_rna=ov.pp.preprocess(adata_rna,mode='shiftlog|pearson',n_HVGs=3000,
                       target_sum=50*1e4)
adata_rna.raw = adata_rna
adata_rna = adata_rna[:, adata_rna.var.highly_variable_features]
ov.pp.scale(adata_rna)
ov.pp.pca(adata_rna,layer='scaled',n_pcs=50)
ov.pp.neighbors(adata_rna, n_neighbors=15, n_pcs=50,
               use_rep='scaled|original|X_pca')
ov.pp.umap(adata_rna)


In [ ]:
from matplotlib import patheffects
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(4,4))

ov.pl.embedding(adata_rna,
                  basis='X_umap',
                  color=['cell_type'],
                   show=False, legend_loc=None, add_outline=False, 
                   frameon='small',legend_fontoutline=2,ax=ax
                 )

ov.utils.gen_mpl_labels(
    adata_rna,
    'cell_type',
    exclude=("None",),  
    basis='X_umap',
    ax=ax,
    adjust_kwargs=dict(arrowprops=dict(arrowstyle='-', color='black')),
    text_kwargs=dict(fontsize= 12 ,weight='bold',
                     path_effects=[patheffects.withStroke(linewidth=2, foreground='w')] ),
)

In [ ]:
for celltype in set(adata_rna.obs['cell_type']):
    print(f"The number of cell type {celltype} is {len(adata_rna[adata_rna.obs['cell_type']==celltype].obs_names)}.")

In [ ]:
simulated_dataset = generate_simulated_dataset(scenario="ggblocks", nside=36, nzprob_nsp=0.2, bkg_mean=0.2, nb_shape=10.0, seed=101)

print(simulated_dataset)

max_indices = np.argmax(simulated_dataset.obsm['spfac'], axis=1)

series = pd.Series(max_indices)

simulated_dataset.obs['spatial_domain'] = max_indices.tolist()
simulated_dataset.obs['spatial_domain'] = simulated_dataset.obs['spatial_domain'].astype('category')

sc.pl.spatial(simulated_dataset,color=['spatial_domain'],spot_size=0.12)

In [ ]:
for spot in set(simulated_dataset.obs['spatial_domain']):
    print(f"The number of spatial domain {spot} is {len(simulated_dataset[simulated_dataset.obs['spatial_domain']==spot].obs_names)}.")

In [ ]:
import scanpy as sc
import numpy as np

np.random.seed(101)


# Specify the cell types to sample and the number of cells for each type
num_cells_per_celltype = {
    'E6Tle4': 612,
    'E5Parm1': 288,
    'E5Sulf1': 216,
    'OliM': 180
}

# Mapping of cluster labels to cell type annotations
cluster2annotation = {
    0: 'E6Tle4',
    1: 'E5Parm1',
    2: 'E5Sulf1',
    3: 'OliM'
}

# Assign cell types to the simulated dataset based on cluster labels
simulated_dataset.obs['cell_type'] = simulated_dataset.obs['spatial_domain'].map(cluster2annotation).astype('category')

# Initialize an empty list to store the indices of selected cells
selected_indices = []

# Create a dictionary to store spatial coordinates for each cell type
celltype_coords = {}

# Iterate over each cell type and randomly sample the specified number of cells
for cell_type in num_cells_per_celltype.keys():
    # Get the indices of cells belonging to the current cell type
    indices = np.where(adata_rna.obs['cell_type'] == cell_type)[0]
    
    # Randomly sample the specified number of cell indices
    selected_indices.extend(np.random.choice(indices, num_cells_per_celltype[cell_type], replace=False))

    # Get the spatial coordinates for cells of the current cell type
    coords = simulated_dataset[simulated_dataset.obs['cell_type'] == cell_type].obsm['spatial']
    celltype_coords[cell_type] = coords

# Initialize an empty array to store the mapped spatial coordinates
mapped_coords = np.zeros((simulated_dataset.n_obs, 2))

# Convert the selected cell indices to a boolean mask
selected_mask = np.in1d(np.arange(adata_rna.n_obs), selected_indices)

# Create a new AnnData object based on the selected cells
new_adata = adata_rna[selected_mask]

# Iterate over the cells in the new AnnData object and map their spatial coordinates
celltype_index_counters = {cell_type: 0 for cell_type in simulated_dataset.obs['cell_type'].unique()}

for i, cell_type in enumerate(new_adata.obs['cell_type']):
    index = celltype_index_counters[cell_type]
    mapped_coords[i] = celltype_coords[cell_type][index]
    celltype_index_counters[cell_type] += 1

# Add the mapped spatial coordinates to the new AnnData object's obsm
new_adata.obsm['spatial'] = mapped_coords

# Create a new AnnData object with the mapped spatial coordinates and UMAP embeddings
adata_rna_new = new_adata.
adata_rna_new.obsm['X_umap'] = adata_rna[adata_rna_new.obs_names, :].obsm['X_umap']

In [ ]:
from matplotlib import patheffects
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(4,4))

ov.pl.embedding(adata_rna_new,
                  basis='X_umap',
                  color=['cell_type'],
                   show=False, legend_loc=None, add_outline=False, 
                   frameon='small',legend_fontoutline=2,ax=ax
                 )

ov.utils.gen_mpl_labels(
    adata_rna_new,
    'cell_type',
    exclude=("None",),  
    basis='X_umap',
    ax=ax,
    adjust_kwargs=dict(arrowprops=dict(arrowstyle='-', color='black')),
    text_kwargs=dict(fontsize= 12 ,weight='bold',
                     path_effects=[patheffects.withStroke(linewidth=2, foreground='w')] ),
)

In [ ]:
adata_rna_new.obs_names = [s[:-3] for s in adata_rna_new.obs_names]

adata_atac = sc.read_h5ad('Original_SingleCell_Multiomics_Data/Dataset_1/Chen-2019-ATAC.h5ad')
adata_atac_new.obsm['spatial'] = adata_rna_new.obsm['spatial']

adata_atac.obs_names = [s[:-4] for s in adata_atac.obs_names]
adata_atac_new = adata_atac[adata_rna_new.obs_names,:]
adata_atac_new.obsm['spatial'] = adata_rna_new.obsm['spatial']
adata_atac_new.obs

In [ ]:
sc.pl.spatial(adata_rna_new,color=['cell_type'],spot_size=0.12)

In [ ]:
adata_rna_new.write_h5ad('Original_Simulated_Data/Simulated_Dataset_3/SimulatedData_3_rna.h5ad',compression='gzip')
adata_atac_new.write_h5ad('Original_Simulated_Data/Simulated_Dataset_3/SimulatedData_3_atac.h5ad',compression='gzip')

# Simulated_data_4

In [ ]:
adata_rna = sc.read_h5ad('Original_SingleCell_Multiomics_Data/Dataset_2/cd34_multiome_rna.h5ad')
adata_rna.X.max()

In [ ]:
adata_rna.layers['raw'] = adata_rna.X
adata_rna=ov.pp.preprocess(adata_rna,mode='shiftlog|pearson',n_HVGs=3000,
                       target_sum=50*1e4)
adata_rna.raw = adata_rna
adata_rna = adata_rna[:, adata_rna.var.highly_variable_features]
ov.pp.scale(adata_rna)
ov.pp.pca(adata_rna,layer='scaled',n_pcs=50)
adata_rna

In [ ]:
from matplotlib import patheffects
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(4,4))

ov.pl.embedding(adata_rna,
                  basis='X_umap',
                  color=['celltype'],
                   show=False, legend_loc=None, add_outline=False, 
                   frameon='small',legend_fontoutline=2,ax=ax
                 )

ov.utils.gen_mpl_labels(
    adata_rna,
    'celltype',
    exclude=("None",),  
    basis='X_umap',
    ax=ax,
    adjust_kwargs=dict(arrowprops=dict(arrowstyle='-', color='black')),
    text_kwargs=dict(fontsize= 12 ,weight='bold',
                     path_effects=[patheffects.withStroke(linewidth=2, foreground='w')] ),
)

In [ ]:
for celltype in set(adata_rna.obs['celltype']):
    print(f"The number of cell type {celltype} is {len(adata_rna[adata_rna.obs['celltype']==celltype].obs_names)}.")

In [ ]:
simulated_dataset = generate_simulated_dataset(scenario="ggblocks", nside=36, nzprob_nsp=0.2, bkg_mean=0.2, nb_shape=10.0, seed=101)

print(simulated_dataset)

max_indices = np.argmax(simulated_dataset.obsm['spfac'], axis=1)

series = pd.Series(max_indices)

simulated_dataset.obs['spatial_domain'] = max_indices.tolist()
simulated_dataset.obs['spatial_domain'] = simulated_dataset.obs['spatial_domain'].astype('category')

sc.pl.spatial(simulated_dataset,color=['spatial_domain'],spot_size=0.12)

In [ ]:
for spot in set(simulated_dataset.obs['spatial_domain']):
    print(f"The number of spatial domain {spot} is {len(simulated_dataset[simulated_dataset.obs['spatial_domain']==spot].obs_names)}.")

In [ ]:
import scanpy as sc
import numpy as np

np.random.seed(101)


# Specify the cell types to sample and the number of cells for each type
num_cells_per_celltype = {
    'HSC': 612,
    'HMP': 288,
    'CLP': 216,
    'Mono': 180
}

# Mapping of cluster labels to cell type annotations
cluster2annotation = {
    0: 'HSC',
    1: 'HMP',
    2: 'CLP',
    3: 'Mono'
}

# Assign cell types to the simulated dataset based on cluster labels
simulated_dataset.obs['cell_type'] = simulated_dataset.obs['spatial_domain'].map(cluster2annotation).astype('category')

# Initialize an empty list to store the indices of selected cells
selected_indices = []

# Create a dictionary to store spatial coordinates for each cell type
celltype_coords = {}

# Iterate over each cell type and randomly sample the specified number of cells
for cell_type in num_cells_per_celltype.keys():
    # Get the indices of cells belonging to the current cell type
    indices = np.where(adata_rna.obs['celltype'] == cell_type)[0]
    
    # Randomly sample the specified number of cell indices
    selected_indices.extend(np.random.choice(indices, num_cells_per_celltype[cell_type], replace=False))

    # Get the spatial coordinates for cells of the current cell type
    coords = simulated_dataset[simulated_dataset.obs['cell_type'] == cell_type].obsm['spatial']
    celltype_coords[cell_type] = coords

# Initialize an empty array to store the mapped spatial coordinates
mapped_coords = np.zeros((simulated_dataset.n_obs, 2))

# Convert the selected cell indices to a boolean mask
selected_mask = np.in1d(np.arange(adata_rna.n_obs), selected_indices)

# Create a new AnnData object based on the selected cells
new_adata = adata_rna[selected_mask]

# Iterate over the cells in the new AnnData object and map their spatial coordinates
celltype_index_counters = {cell_type: 0 for cell_type in simulated_dataset.obs['cell_type'].unique()}

for i, cell_type in enumerate(new_adata.obs['celltype']):
    index = celltype_index_counters[cell_type]
    mapped_coords[i] = celltype_coords[cell_type][index]
    celltype_index_counters[cell_type] += 1

# Add the mapped spatial coordinates to the new AnnData object's obsm
new_adata.obsm['spatial'] = mapped_coords

# Create a new AnnData object with the mapped spatial coordinates and UMAP embeddings
adata_rna_new = new_adata.copy()
adata_rna_new.obsm['X_umap'] = adata_rna[adata_rna_new.obs_names, :].obsm['X_umap']
adata_rna_new.obs['cell_type'] = adata_rna_new.obs['celltype']
adata_rna_new

In [ ]:
from matplotlib import patheffects
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(4,4))

ov.pl.embedding(adata_rna_new,
                  basis='X_umap',
                  color=['cell_type'],
                   show=False, legend_loc=None, add_outline=False, 
                   frameon='small',legend_fontoutline=2,ax=ax
                 )

ov.utils.gen_mpl_labels(
    adata_rna_new,
    'cell_type',
    exclude=("None",),  
    basis='X_umap',
    ax=ax,
    adjust_kwargs=dict(arrowprops=dict(arrowstyle='-', color='black')),
    text_kwargs=dict(fontsize= 12 ,weight='bold',
                     path_effects=[patheffects.withStroke(linewidth=2, foreground='w')] ),
)

In [ ]:
adata_atac = sc.read_h5ad('Original_SingleCell_Multiomics_Data/Dataset_2/cd34_multiome_atac.h5ad')
adata_atac_new = adata_atac[adata_rna_new.obs_names,:]
adata_atac_new.obs

In [ ]:
adata_rna_new.write_h5ad('Original_Simulated_Data/Simulated_Dataset_4/SimulatedData_4_rna.h5ad',compression='gzip')
adata_atac_new.write_h5ad('Original_Simulated_Data/Simulated_Dataset_4/SimulatedData_4_atac.h5ad',compression='gzip')

# Simulated_data_5

In [ ]:
adata_rna = sc.read_h5ad('Original_SingleCell_Multiomics_Data/Dataset_2/cd34_multiome_rna.h5ad')
adata_rna.X.max()

In [ ]:
adata_rna.layers['raw'] = adata_rna.X
adata_rna=ov.pp.preprocess(adata_rna,mode='shiftlog|pearson',n_HVGs=3000,
                       target_sum=50*1e4)
adata_rna.raw = adata_rna
adata_rna = adata_rna[:, adata_rna.var.highly_variable_features]
ov.pp.scale(adata_rna)
ov.pp.pca(adata_rna,layer='scaled',n_pcs=50)
adata_rna

In [ ]:
from matplotlib import patheffects
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(4,4))

ov.pl.embedding(adata_rna,
                  basis='X_umap',
                  color=['celltype'],
                   show=False, legend_loc=None, add_outline=False, 
                   frameon='small',legend_fontoutline=2,ax=ax
                 )

ov.utils.gen_mpl_labels(
    adata_rna,
    'celltype',
    exclude=("None",),  
    basis='X_umap',
    ax=ax,
    adjust_kwargs=dict(arrowprops=dict(arrowstyle='-', color='black')),
    text_kwargs=dict(fontsize= 12 ,weight='bold',
                     path_effects=[patheffects.withStroke(linewidth=2, foreground='w')] ),
)

In [ ]:
for celltype in set(adata_rna.obs['celltype']):
    print(f"The number of cell type {celltype} is {len(adata_rna[adata_rna.obs['celltype']==celltype].obs_names)}.")

In [ ]:
simulated_dataset = generate_simulated_dataset(scenario="ggblocks", nside=36, nzprob_nsp=0.2, bkg_mean=0.2, nb_shape=10.0, seed=101)

print(simulated_dataset)

max_indices = np.argmax(simulated_dataset.obsm['spfac'], axis=1)

series = pd.Series(max_indices)

simulated_dataset.obs['spatial_domain'] = max_indices.tolist()
simulated_dataset.obs['spatial_domain'] = simulated_dataset.obs['spatial_domain'].astype('category')

sc.pl.spatial(simulated_dataset,color=['spatial_domain'],spot_size=0.12)

In [ ]:
for spot in set(simulated_dataset.obs['spatial_domain']):
    print(f"The number of spatial domain {spot} is {len(simulated_dataset[simulated_dataset.obs['spatial_domain']==spot].obs_names)}.")

In [ ]:
import scanpy as sc
import numpy as np

np.random.seed(101)


# Specify the cell types to sample and the number of cells for each type
num_cells_per_celltype = {
    'HSC': 612,
    'HMP': 288,
    'MEP': 216,
    'Ery': 180
}

# Mapping of cluster labels to cell type annotations
cluster2annotation = {
    0: 'HSC',
    1: 'HMP',
    2: 'MEP',
    3: 'Ery'
}

# Assign cell types to the simulated dataset based on cluster labels
simulated_dataset.obs['cell_type'] = simulated_dataset.obs['spatial_domain'].map(cluster2annotation).astype('category')

# Initialize an empty list to store the indices of selected cells
selected_indices = []

# Create a dictionary to store spatial coordinates for each cell type
celltype_coords = {}

# Iterate over each cell type and randomly sample the specified number of cells
for cell_type in num_cells_per_celltype.keys():
    # Get the indices of cells belonging to the current cell type
    indices = np.where(adata_rna.obs['celltype'] == cell_type)[0]
    
    # Randomly sample the specified number of cell indices
    selected_indices.extend(np.random.choice(indices, num_cells_per_celltype[cell_type], replace=False))

    # Get the spatial coordinates for cells of the current cell type
    coords = simulated_dataset[simulated_dataset.obs['cell_type'] == cell_type].obsm['spatial']
    celltype_coords[cell_type] = coords

# Initialize an empty array to store the mapped spatial coordinates
mapped_coords = np.zeros((simulated_dataset.n_obs, 2))

# Convert the selected cell indices to a boolean mask
selected_mask = np.in1d(np.arange(adata_rna.n_obs), selected_indices)

# Create a new AnnData object based on the selected cells
new_adata = adata_rna[selected_mask]

# Iterate over the cells in the new AnnData object and map their spatial coordinates
celltype_index_counters = {cell_type: 0 for cell_type in simulated_dataset.obs['cell_type'].unique()}

for i, cell_type in enumerate(new_adata.obs['celltype']):
    index = celltype_index_counters[cell_type]
    mapped_coords[i] = celltype_coords[cell_type][index]
    celltype_index_counters[cell_type] += 1

# Add the mapped spatial coordinates to the new AnnData object's obsm
new_adata.obsm['spatial'] = mapped_coords

# Create a new AnnData object with the mapped spatial coordinates and UMAP embeddings
adata_rna_new = new_adata.copy()
adata_rna_new.obsm['X_umap'] = adata_rna[adata_rna_new.obs_names, :].obsm['X_umap']
adata_rna_new.obs['cell_type'] = adata_rna_new.obs['celltype']
adata_rna_new

In [ ]:
from matplotlib import patheffects
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(4,4))

ov.pl.embedding(adata_rna_new,
                  basis='X_umap',
                  color=['cell_type'],
                   show=False, legend_loc=None, add_outline=False, 
                   frameon='small',legend_fontoutline=2,ax=ax
                 )

ov.utils.gen_mpl_labels(
    adata_rna_new,
    'cell_type',
    exclude=("None",),  
    basis='X_umap',
    ax=ax,
    adjust_kwargs=dict(arrowprops=dict(arrowstyle='-', color='black')),
    text_kwargs=dict(fontsize= 12 ,weight='bold',
                     path_effects=[patheffects.withStroke(linewidth=2, foreground='w')] ),
)

In [ ]:
adata_atac = sc.read_h5ad('Original_SingleCell_Multiomics_Data/Dataset_2/cd34_multiome_atac.h5ad')
adata_atac_new = adata_atac[adata_rna_new.obs_names,:]
adata_atac_new.obs

In [ ]:
adata_rna_new.write_h5ad('Original_Simulated_Data/Simulated_Dataset_5/SimulatedData_5_rna.h5ad',compression='gzip')
adata_atac_new.write_h5ad('Original_Simulated_Data/Simulated_Dataset_5/SimulatedData_5_atac.h5ad',compression='gzip')

# Test

In [ ]:
adata_rna_1 = sc.read_h5ad('Original_Simulated_Data/Simulated_Dataset_1/SimulatedData_1_rna.h5ad')
sc.pl.spatial(adata_rna_1,color=['cell_type'],spot_size=0.12)

In [ ]:
adata_rna_1 = sc.read_h5ad('Original_Simulated_Data/Simulated_Dataset_2/SimulatedData_2_rna.h5ad')
sc.pl.spatial(adata_rna_1,color=['cell_type'],spot_size=0.12)

In [ ]:
adata_rna_1 = sc.read_h5ad('Original_Simulated_Data/Simulated_Dataset_3/SimulatedData_3_rna.h5ad')
sc.pl.spatial(adata_rna_1,color=['cell_type'],spot_size=0.12)

In [ ]:
adata_rna_1 = sc.read_h5ad('Original_Simulated_Data/Simulated_Dataset_4/SimulatedData_4_rna.h5ad')
sc.pl.spatial(adata_rna_1,color=['cell_type'],spot_size=0.12)

In [ ]:
adata_rna_1 = sc.read_h5ad('Original_Simulated_Data/Simulated_Dataset_5/SimulatedData_5_rna.h5ad')
sc.pl.spatial(adata_rna_1,color=['cell_type'],spot_size=0.12)